# THE VEHICLE ROUTING PROBLEM
## Case Study : Indomaret and Alfamart (Surabaya, Indonesia)

The Vehicle Routing Problem (VRP) is a classic logistics and operations research problem. Its goal is to determine the most cost-effective routes for a fleet of vehicles to service a given set of locations (customers), starting and ending at a central depot. It has become well-known because of its usage in various industries. The FMCG industry consistently faces this challenge because of its need for rapid distribution. In Indonesia, The fast-moving consumer goods (FMCG) market is on rapid ascent. The products itself is distributed through both modern channels such as convenience stores and traditional channels such as warungs or mom-and-pop shops. 

In this notebook, I explore VRP using a case study of product distribution in convenience stores such as Indomaret and Alfamart in the Surabaya Area. The data was scraped from Overpass for the accuracy of the store locations. In total, there are 96 Indomaret and Alfamart that have been successfully gathered. This notebook aims to optimize the delivery routes for 96 convenience stores across Surabaya using the Capacitated VRP (CVRP) approach.

## Problem Formulation
This study formulates the distribution problem as a Capacitated Vehicle Routing Problem (CVRP). In this formulation, there are total of 97 nodes with 96 nodes representing Indomaret or Alfamart. Node 0 represents the depot which the vehicles must return to. Each store is assigned a demand of 1 unit, representing one delivery drop off per store. For this case study, each truck has a capacity of 10 units. In total, there are 10 trucks. The objective is to minimize the total distance traveled.

# Data Collection

The store location data was collected with Overpass Turbo via browser with a GeoJSON format. The query is shown below:

[out:json][timeout:60];
area[name="Surabaya"][admin_level=5]->.searchArea;
node["brand"~"Indomaret|Alfamart"]["shop"="convenience"](area.searchArea);
out body;

Overpass was chosen as the data source due to its open source nature. It provides real-world store coordinates without requiring proprietary APIs.

In [1]:
import json
import pandas as pd

with open('data.geojson', 'r') as f:
    geojson = json.load(f)

rows = []
for feature in geojson['features']:
    coords = feature['geometry']['coordinates']
    props = feature['properties']
    rows.append({
        'id': feature['id'],
        'brand': props.get('brand'),
        'lat': coords[1],
        'lon': coords[0],
        'address': props.get('addr:street', 'N/A')
    })

df = pd.DataFrame(rows)

print(f"Total of minimarkets: {len(df)}")
print(f"Indomaret: {len(df[df['brand'] == 'Indomaret'])}")
print(f"Alfamart: {len(df[df['brand'] == 'Alfamart'])}")
print()
print(df.head())

Total of minimarkets: 96
Indomaret: 63
Alfamart: 33

                id      brand       lat         lon address
0  node/1359471683  Indomaret -7.314559  112.789765     N/A
1  node/2989728339  Indomaret -7.311784  112.746417     N/A
2  node/2989728395  Indomaret -7.308946  112.744605     N/A
3  node/3067860977   Alfamart -7.301474  112.775559     N/A
4  node/3121609244  Indomaret -7.268070  112.796515     N/A


## Data Pre-processing

The depot is placed around the Margomulyo Industrial Area. This is because it is known that the Margomulyo Industrial Area accomodate a lot of warehouses inside the area.

In [2]:
depot = {
    'id': 'depot',
    'brand': 'DEPOT',
    'lat': -7.2372,
    'lon': 112.6878,
    'address': 'Margomulyo Industrial Area'
}

depot_df = pd.DataFrame([depot])
df_full = pd.concat([depot_df, df], ignore_index=True)

print(df_full.head(3))
print(f"\nTotal nodes (depot + customers): {len(df_full)}")

                id      brand       lat         lon  \
0            depot      DEPOT -7.237200  112.687800   
1  node/1359471683  Indomaret -7.314559  112.789765   
2  node/2989728339  Indomaret -7.311784  112.746417   

                      address  
0  Margomulyo Industrial Area  
1                         N/A  
2                         N/A  

Total nodes (depot + customers): 97


To better understand the spread of all 96 Indomaret and Alfamart in Surabaya, Folium is used to display the visual location of all 97 nodes including the Depot.

In [3]:
pip install folium

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import folium

m = folium.Map(location=[-7.28, 112.73], zoom_start=12)

# Depot Plotting
folium.Marker(
    location=[depot['lat'], depot['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m)

# Minimarkets Plotting
for _, row in df.iterrows():
    color = 'blue' if row['brand'] == 'Indomaret' else 'green'
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=5,
        color=color,
        fill=True,
        popup=row['brand']
    ).add_to(m)

m.save('minimarket_surabaya.html')
print("Map saved!")

Map saved!


![alt text](minimarket.png)

Based on the map displayed with Folium, the spread is denser in the center of Surabaya and become sparser in the western skirt of the City. Indomaret (blue) is more dominant and spread more evenly rather than Alfamart (green). The depot is located in the northern part of the city, slightly farther from the majority of stores which are concentrated in the center.

In [5]:
import numpy as np

# Assign demand
df['demand'] = 1

NUM_VEHICLES = 10
VEHICLE_CAPACITY = 10
DEPOT_INDEX = 0

print(df_full.head())
print(f"\nTotal demand: {df['demand'].sum()}")
print(f"Vehicles: {NUM_VEHICLES}, Capacity each: {VEHICLE_CAPACITY}")
print(f"Max total capacity: {NUM_VEHICLES * VEHICLE_CAPACITY}")

                id      brand       lat         lon  \
0            depot      DEPOT -7.237200  112.687800   
1  node/1359471683  Indomaret -7.314559  112.789765   
2  node/2989728339  Indomaret -7.311784  112.746417   
3  node/2989728395  Indomaret -7.308946  112.744605   
4  node/3067860977   Alfamart -7.301474  112.775559   

                      address  
0  Margomulyo Industrial Area  
1                         N/A  
2                         N/A  
3                         N/A  
4                         N/A  

Total demand: 96
Vehicles: 10, Capacity each: 10
Max total capacity: 100


To calculate the distance between two coordinates, the Haversine formula is used 
instead of the Euclidean distance. This is because the Earth is not flat. Euclidean distance treats coordinates as points on a 2D plane, which introduces 
error when dealing with real-world geographic coordinates. The Haversine formula 
accounts for the curvature of the Earth, where R = 6371 represents the Earth's 
radius in kilometers. Since trigonometric functions in Python require input in 
radians, the coordinates are first converted from degrees using `map(radians, ...)` 
before the calculation is performed.

In [6]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371

    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))

    return R * c

# Test: Depot's distance to the nearest minimarket
depot_row = df_full.iloc[0]
toko_pertama = df_full.iloc[1]

jarak = haversine(depot_row['lat'], depot_row['lon'],
                  toko_pertama['lat'], toko_pertama['lon'])
print(f"Distance from depot to the first store: {jarak:.2f} km")

Distance from depot to the first store: 14.16 km


In [7]:
import numpy as np

all_points = df_full[['lat', 'lon']].values
n = len(all_points)

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i != j:
            distance_matrix[i][j] = haversine(
                all_points[i][0], all_points[i][1],
                all_points[j][0], all_points[j][1]
            )

print(f"Shape matrix: {distance_matrix.shape}")
print(f"\n Depot distance towards the first five stores (Example)")
print(distance_matrix[0][:6].round(2))

Shape matrix: (97, 97)

 Depot distance towards the first five stores (Example)
[ 0.   14.16 10.52 10.14 12.03 12.47]


## Nearest Neighbour Heuristic Algorithm
The Nearest Neighbour heuristic is implemented to generate a baseline solution. 
Starting from the depot, each vehicle greedily visits the nearest unvisited store 
until its capacity is full, then returns to the depot and the next vehicle begins. 
This continues until all 96 stores are assigned to a route.

In [8]:
unvisited = list(range(1, len(df_full)))
routes = []

while unvisited:
    route = []
    load = 0
    pos = 0

    while True:
        if not unvisited:
            break

        nearest = min(unvisited, key=lambda x: distance_matrix[pos][x])

        if load + 1 <= VEHICLE_CAPACITY:
            route.append(nearest)
            unvisited.remove(nearest)
            load += 1
            pos = nearest
        else:
            break

    routes.append(route)

print(f"Total routes: {len(routes)}")
for i, r in enumerate(routes):
    print(f"Truck {i+1}: {len(r)} store(s)")

Total routes: 10
Truck 1: 10 store(s)
Truck 2: 10 store(s)
Truck 3: 10 store(s)
Truck 4: 10 store(s)
Truck 5: 10 store(s)
Truck 6: 10 store(s)
Truck 7: 10 store(s)
Truck 8: 10 store(s)
Truck 9: 10 store(s)
Truck 10: 6 store(s)


Next, we calculate the total distance from the result of using the Nearest Neighbour heuristic.

In [9]:
def hitung_total_jarak(routes, distance_matrix):
    total = 0
    for route in routes:
        pos = 0
        for toko in route:
            total += distance_matrix[pos][toko]
            pos = toko
        total += distance_matrix[pos][0]
    return total

total_jarak = hitung_total_jarak(routes, distance_matrix)
print(f"Total distances of all trucks: {total_jarak:.2f} km")

for i, route in enumerate(routes):
    pos = 0
    jarak = 0
    for toko in route:
        jarak += distance_matrix[pos][toko]
        pos = toko
    jarak += distance_matrix[pos][0]
    print(f"Truck {i+1}: {jarak:.2f} km | {len(route)} store(s)")

Total distances of all trucks: 277.52 km
Truck 1: 19.33 km | 10 store(s)
Truck 2: 15.70 km | 10 store(s)
Truck 3: 16.58 km | 10 store(s)
Truck 4: 26.71 km | 10 store(s)
Truck 5: 47.33 km | 10 store(s)
Truck 6: 20.71 km | 10 store(s)
Truck 7: 24.92 km | 10 store(s)
Truck 8: 27.98 km | 10 store(s)
Truck 9: 31.15 km | 10 store(s)
Truck 10: 47.10 km | 6 store(s)


To better understand the performance of using Nearest Neighbour heuristic, the result is displayed using Folium. 

In [10]:
import folium

colors = ['red','blue','green','purple','orange',
          'darkred','darkblue','darkgreen','cadetblue','black']

m = folium.Map(location=[-7.28, 112.73], zoom_start=12)

folium.Marker(
    location=[df_full.iloc[0]['lat'], df_full.iloc[0]['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m)

for i, route in enumerate(routes):
    coords = [df_full.iloc[0][['lat','lon']].tolist()]
    for idx in route:
        row = df_full.iloc[idx]
        coords.append([row['lat'], row['lon']])
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=colors[i],
            fill=True,
            popup=f"Truk {i+1} | {row['brand']}"
        ).add_to(m)
    coords.append(df_full.iloc[0][['lat','lon']].tolist())

    folium.PolyLine(coords, color=colors[i], weight=2).add_to(m)

m.save('vrp_routes.html')
print("Map saved!")

Map saved!


![alt text](nearestneighbour.png)

The Nearest Neighbour heuristic produces a total distance of 277.52 km across 
10 trucks. However, as visible on the map, the routes are uneven, early trucks 
claim nearby stores first, leaving later trucks to cover distant areas. This is 
evident with Truck 5 and Truck 10 traveling 47 km each, while Truck 2 only covers 
15 km. This greedy nature makes Nearest Neighbour a poor optimizer, serving only 
as a baseline for comparison.

## Optimization: OR-Tools

To improve upon the Nearest Neighbour baseline, Google OR-Tools is used to solve 
the CVRP. OR-Tools is an open-source optimization library developed by Google, 
widely used in industry for solving routing, scheduling, and integer programming 
problems.

In [11]:
from sklearn.cluster import KMeans

coords = df[['lat', 'lon']].values

kmeans = KMeans(n_clusters=10, random_state=42)
df['cluster'] = kmeans.fit_predict(coords)

print(df['cluster'].value_counts().sort_index())

cluster
0     2
1    15
2    10
3    15
4    10
5    11
6     9
7     3
8    14
9     7
Name: count, dtype: int64


In [12]:
pip install ortools

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

In [14]:
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

def create_data_model():
    data = {}

    data['distance_matrix'] = (distance_matrix * 100).astype(int).tolist()
    data['demands'] = [0] + [1] * len(df)
    data['vehicle_capacities'] = [VEHICLE_CAPACITY] * NUM_VEHICLES
    data['num_vehicles'] = NUM_VEHICLES
    data['depot'] = 0

    return data

data = create_data_model()

print(f"Total nodes: {len(data['distance_matrix'])}")
print(f"Total vehicles: {data['num_vehicles']}")
print(f"Vehicle capacities: {data['vehicle_capacities'][0]}")
print(f"Total demand: {sum(data['demands'])}")

Total nodes: 97
Total vehicles: 10
Vehicle capacities: 10
Total demand: 96


In [15]:
manager = pywrapcp.RoutingIndexManager(
    len(data['distance_matrix']),
    data['num_vehicles'],
    data['depot']
)

routing = pywrapcp.RoutingModel(manager)

def distance_callback(from_index, to_index):
    from_node = manager.IndexToNode(from_index)
    to_node = manager.IndexToNode(to_index)
    return data['distance_matrix'][from_node][to_node]

transit_callback_index = routing.RegisterTransitCallback(distance_callback)
routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

def demand_callback(from_index):
    from_node = manager.IndexToNode(from_index)
    return data['demands'][from_node]

demand_callback_index = routing.RegisterUnaryTransitCallback(demand_callback)
routing.AddDimensionWithVehicleCapacity(
    demand_callback_index,
    0,
    data['vehicle_capacities'],
    True,
    'Capacity'
)

search_params = pywrapcp.DefaultRoutingSearchParameters()
search_params.first_solution_strategy = (
    routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC
)

solution = routing.SolveWithParameters(search_params)

if solution:
    print("Solution found!")
else:
    print("Solution not found.")

Solution found!


In [16]:
def print_solution(data, manager, routing, solution):
    total_jarak = 0
    routes_ortools = []

    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route = []
        jarak = 0

        while not routing.IsEnd(index):
            node = manager.IndexToNode(index)
            if node != data['depot']:
                route.append(node)
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            jarak += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        jarak_km = jarak / 100
        total_jarak += jarak_km
        routes_ortools.append(route)
        print(f"Truk {vehicle_id+1}: {len(route)} toko | {jarak_km:.2f} km")

    print(f"\nTotal distance using OR-Tools: {total_jarak:.2f} km")
    return routes_ortools

routes_ortools = print_solution(data, manager, routing, solution)

Truk 1: 10 toko | 36.18 km
Truk 2: 10 toko | 28.53 km
Truk 3: 10 toko | 26.57 km
Truk 4: 10 toko | 22.30 km
Truk 5: 10 toko | 21.03 km
Truk 6: 10 toko | 15.52 km
Truk 7: 9 toko | 18.51 km
Truk 8: 10 toko | 25.99 km
Truk 9: 10 toko | 29.37 km
Truk 10: 7 toko | 14.58 km

Total distance using OR-Tools: 238.58 km


OR-Tools is fed with the distance matrix, demand, vehicle capacities, and depot 
location to solve the CVRP. The solver finds the most cost-efficient routes 
automatically, resulting in a total distance of 238.58 km, which is 38.94 km (14%) less 
than the Nearest Neighbour baseline. The routes are also more geographically 
clustered, with each truck covering stores in the same area.

For better visualization, the result is displayed with Folium.

In [17]:
m2 = folium.Map(location=[-7.28, 112.73], zoom_start=12)

# Depot
folium.Marker(
    location=[df_full.iloc[0]['lat'], df_full.iloc[0]['lon']],
    popup='DEPOT',
    icon=folium.Icon(color='red', icon='home')
).add_to(m2)

for i, route in enumerate(routes_ortools):
    coords = [df_full.iloc[0][['lat','lon']].tolist()]
    for idx in route:
        row = df_full.iloc[idx]
        coords.append([row['lat'], row['lon']])
        folium.CircleMarker(
            location=[row['lat'], row['lon']],
            radius=5,
            color=colors[i],
            fill=True,
            popup=f"Truk {i+1} | {row['brand']}"
        ).add_to(m2)
    coords.append(df_full.iloc[0][['lat','lon']].tolist())
    folium.PolyLine(coords, color=colors[i], weight=2).add_to(m2)

m2.save('vrp_routes_ortools.html')
print("Map saved!")

Map saved!


![alt text](or_tools.png)

## Conclusion & Insight

This project explores the Vehicle Routing Problem (CVRP) using real store location 
data from Surabaya. Two approaches were compared:

- **Nearest Neighbour**: 277.52 km (baseline)
- **OR-Tools**: 238.58 km (optimized)

OR-Tools managed to cut the total distance by 38.94 km, or around 14%. In a 
real-world setting, this translates to lower fuel costs and faster deliveries.

A few limitations to note:
- Demand is simplified to 1 unit per store, which may not reflect reality
- Distance is calculated as straight-line (Haversine), not actual road distance
- The depot location is assumed, not based on real warehouse data

As a next step, road network distance using OSRM or Google Maps API could replace 
Haversine for more realistic results.